In [ ]:
#| default_exp drive

In [ ]:
#| hide
import string
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Run each deployment operation as a separate library call.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import json, sys

In [ ]:
#| export
from dataclasses import dataclass, field

In [ ]:
#| export
from pathlib import Path

In [ ]:
#| export
from pullup.env import EnvStore

In [ ]:
#| export
from pullup.env import env_value

In [ ]:
#| export
from pullup.pipeline import DIR, Pipeline, PipelineError

In [ ]:
#| export
from pullup.project import Step

In [ ]:
#| export
DRIVE_FILE = 'drive.json'

In [ ]:
#| export
DriveError = PipelineError

Drive plans are stored separately in `.pullup/drive.json`.

In [ ]:
DRIVE_FILE, DriveError is PipelineError

('drive.json', True)

In [ ]:
#| hide
tmp = TemporaryDirectory(); root = Path(tmp.name).resolve()/'demo'; root.mkdir()
(root/'pyproject.toml').write_text('[project]\nname = "demo"\n')
class Env:
    "An environment store holding the values it was given, in place of the keychain `EnvStore` reads."
    def __init__(self, **kw): self.d = kw
    def get(self, key, secret=False): return self.d.get(key, '')
    def values(self, keys, secret=True): return {k: self.d[k] for k in keys if k in self.d}
env = Env(DOMAIN='app.example.com', PORT='8080')

In [ ]:
#| export
@dataclass
class Part:
    "One library call, as a step you can run on its own."
    id: str
    package: str
    label: str
    doc: str
    code: str
    needs: list = field(default_factory=list)
    writes: list = field(default_factory=list)
    produces: list = field(default_factory=list)

A `Part` defines one code template and its environment requirements.

In [ ]:
#| export
PARTS = [
    Part('image', 'dockeasy', 'Dockerfile',
         'detect_app reads the project and returns the image it would build. Written here so '
         'the compose stack and the workflow build the same thing.',
         "from dockeasy import detect_app\n"
         "df = detect_app({root!r}, pkgs={pkgs!r}, vols={vols!r})\n"
         "open({root!r} + '/Dockerfile', 'w').write(str(df) + '\\n')\n"
         "print(df)",
         produces=['Dockerfile']),
    Part('stack', 'vpseasy', 'compose stack',
         'caddy_stack wraps that image in Caddy and cloudflared, and writes docker-compose.yml '
         'and the Caddyfile beside it.',
         "from dockeasy import detect_app\n"
         "from vpseasy import caddy_stack\n"
         "df = detect_app({root!r}, pkgs={pkgs!r}, vols={vols!r})\n"
         "c = caddy_stack({host!r}, df, vols={vols!r}, root={root!r})\n"
         "print(c)",
         produces=['docker-compose.yml', 'Caddyfile', 'Dockerfile']),
    Part('tunnel', 'cfeasy', 'Cloudflare tunnel',
         'setup_tunnel reuses the tunnel of this name or creates it, points the hostname at it, '
         'and returns the token the container runs cloudflared with.',
         "from cfeasy import CF\n"
         "from dockeasy import env_get, env_set\n"
         "cf = CF(token=env_get('CLOUDFLARE_API_TOKEN'))\n"
         "tid, tok = cf.setup_tunnel({domain!r}, {subdomain!r} or None, tunnel_name={tunnel!r})\n"
         "env_set('CF_TUNNEL_TOKEN', tok)\n"
         "print('tunnel', tid, 'for', {host!r})",
         needs=['CLOUDFLARE_API_TOKEN'], writes=['CF_TUNNEL_TOKEN']),
    Part('dns', 'cfeasy', 'extra hostname',
         'A second hostname on the same tunnel and the same container. Skipped unless '
         'SECOND_HOST is set; Caddy tells the two apart by the Host header.',
         "from cfeasy import CF\n"
         "from dockeasy import env_get\n"
         "host = env_get('SECOND_HOST', default='')\n"
         "if not host: raise SystemExit('SECOND_HOST is not set — nothing to point')\n"
         "cf = CF(token=env_get('CLOUDFLARE_API_TOKEN'))\n"
         "tid = cf.tunnel_id({tunnel!r})\n"
         "print(cf.tunnel_cname(host, host, tid))",
         needs=['CLOUDFLARE_API_TOKEN']),
    Part('server', 'vpseasy', 'server and rsync',
         'hetzner_deploy provisions the VPS if it is missing, waits for cloud-init, rsyncs the '
         'directory and brings the stack up. Idempotent: run it again after a fix.',
         "from dockeasy import env_get, env_set\n"
         "from vpseasy import hetzner_deploy\n"
         "r = hetzner_deploy(env_get('SERVER_NAME', default={app!r}), {root!r},\n"
         "                   include={inc!r}, exclude={exc!r}, path='/srv/app',\n"
         "                   user=env_get('SERVER_USER', default='deploy'))\n"
         "env_set('HETZNER_IP', r.ip)\n"
         "print('deployed', r.name, 'at', r.ip)",
         needs=['HCLOUD_TOKEN'], writes=['HETZNER_IP']),
    Part('secrets', 'gheasy', 'push secrets',
         'gh_push_env sends every key the schema declares to GitHub, as a secret or a variable '
         'depending on how it is declared, so the workflow can run this same deploy.',
         "from gheasy import gh_push_env\n"
         "from dockeasy import env_get\n"
         "keys = {keys!r}\n"
         "vals = {{k: v for k in keys if (v := env_get(k, default=''))}}\n"
         "print('pushing', len(vals), 'of', len(keys), 'declared keys')\n"
         "gh_push_env(vals, path={root!r})",
         needs=['GITHUB_TOKEN']),
]

The six parts create build files, configure the tunnel and DNS, provision the server, and publish secrets.

In [ ]:
for p in PARTS: print(f'{p.id:8} {p.package:9} {p.label:18} needs={p.needs} writes={p.writes}')

image    dockeasy  Dockerfile         needs=[] writes=[]
stack    vpseasy   compose stack      needs=[] writes=[]
tunnel   cfeasy    Cloudflare tunnel  needs=['CLOUDFLARE_API_TOKEN'] writes=['CF_TUNNEL_TOKEN']
dns      cfeasy    extra hostname     needs=['CLOUDFLARE_API_TOKEN'] writes=[]
server   vpseasy   server and rsync   needs=['HCLOUD_TOKEN'] writes=['HETZNER_IP']
secrets  gheasy    push secrets       needs=['GITHUB_TOKEN'] writes=[]


In [ ]:
#| hide
test_eq(len(PARTS), len({p.id for p in PARTS}))
assert all('CLOUDFLARE_API_TOKEN' in p.needs for p in PARTS if 'CLOUDFLARE_API_TOKEN' in p.code), \
    'a snippet that reads a token declares it, or the state cannot say it is missing'

In [ ]:
#| export
def context(root, env=None):
    "The values every snippet is rendered with, resolved once."
    root = Path(root).expanduser().resolve()
    store = env or EnvStore()
    def value(key, default=''): return env_value(store, key, default)
    app, schema = '', {}
    try:
        from gheasy.core import GheasyConfig
        cfg = GheasyConfig.load(str(root))
        app, schema = cfg.app, dict(cfg.env_schema or {})
    except Exception: pass
    if not app:
        try:
            import tomllib
            with open(root/'pyproject.toml', 'rb') as f:
                app = str(((tomllib.load(f).get('project') or {}).get('name') or '')).strip()
        except (OSError, ValueError, ImportError): pass
    app = app or root.name
    host = value('DOMAIN')
    domain, subdomain = host, ''
    if host.count('.') > 1:
        subdomain, _, domain = host.partition('.')
    return {'root': str(root), 'app': app, 'host': host or root.name, 'domain': domain,
            'subdomain': subdomain, 'tunnel': f'{subdomain or app}_{domain}' if domain else app,
            'port': value('PORT', '5001'), 'keys': sorted(schema) or ['DOMAIN'],
            'pkgs': ['curl'], 'vols': ['/app/data'],
            'inc': [f'{app}/', 'pyproject.toml', 'uv.lock', 'main.py', 'Dockerfile',
                    'docker-compose.yml', 'Caddyfile', '.env'],
            'exc': ['data/', '.git/', '.venv/', '__pycache__/']}

`context` resolves the application name and environment values used to render every part.

In [ ]:
ctx = context(root, env)
{k: ctx[k] for k in ('app', 'host', 'domain', 'subdomain', 'tunnel', 'port', 'keys')}

{'app': 'demo',
 'host': 'app.example.com',
 'domain': 'example.com',
 'subdomain': 'app',
 'tunnel': 'app_example.com',
 'port': '8080',
 'keys': ['DOMAIN']}

In [ ]:
#| hide
fields = {f for p in PARTS for _, f, _, _ in string.Formatter().parse(p.code) if f}
assert fields <= set(ctx), f'rendered with values context does not resolve: {fields - set(ctx)}'
apex = context(root, Env(DOMAIN='example.com'))
test_eq((apex['subdomain'], apex['domain'], apex['tunnel']), ('', 'example.com', 'demo_example.com'))
bare = context(root, Env())
test_eq((bare['host'], bare['domain'], bare['tunnel'], bare['port']), ('demo', '', 'demo', '5001'))
test_eq((bare['keys'], bare['inc'][0]), (['DOMAIN'], 'demo/'))
test_eq(context(root/'gone', Env())['app'], 'gone')

In [ ]:
#| export
def _step(part, exe, ctx):
    "One part as a `Step`, its snippet rendered with this project's values."
    code = part.code.format(**ctx)
    return Step(id=part.id, label=f'{part.package} · {part.label}', doc=part.doc,
                cmd=f'python -c … {part.package}', argv=[exe, '-c', code], needs=list(part.needs),
                meta={'package': part.package, 'code': code, 'writes': list(part.writes),
                      'produces': list(part.produces), 'title': part.label})

`_step` renders a part into a `Step` whose `argv` runs the snippet under Python.

In [ ]:
s = _step(PARTS[2], 'python', ctx)
print(s.label, '|', s.cmd, '|', s.needs)
print(s.meta['code'])

cfeasy · Cloudflare tunnel | python -c … cfeasy | ['CLOUDFLARE_API_TOKEN']
from cfeasy import CF
from dockeasy import env_get, env_set
cf = CF(token=env_get('CLOUDFLARE_API_TOKEN'))
tid, tok = cf.setup_tunnel('example.com', 'app' or None, tunnel_name='app_example.com')
env_set('CF_TUNNEL_TOKEN', tok)
print('tunnel', tid, 'for', 'app.example.com')


In [ ]:
#| export
def parts(root, env=None, python=''):
    "Every part as a runnable step, with this project's values already in the snippet."
    ctx = context(root, env)
    return [_step(p, str(python or sys.executable), ctx) for p in PARTS]

`parts` renders every part with one context and the selected Python interpreter.

In [ ]:
steps = parts(root, env)
test_eq(steps[0].argv[0], sys.executable)
print(steps[-1].meta['code'].replace(str(root), '/proj'))

from gheasy import gh_push_env
from dockeasy import env_get
keys = ['DOMAIN']
vals = {k: v for k in keys if (v := env_get(k, default=''))}
print('pushing', len(vals), 'of', len(keys), 'declared keys')
gh_push_env(vals, path='/proj')


In [ ]:
#| hide
steps = parts(root, env, python='/usr/bin/python3')
test_eq([s.id for s in steps], [p.id for p in PARTS])
for s in steps:
    test_eq(s.argv[:2], ['/usr/bin/python3', '-c'])
    assert '!r}' not in s.argv[2], f'{s.id} reached the child with a placeholder in it'
    compile(s.argv[2], s.id, 'exec')

In [ ]:
#| export
class Drive(Pipeline):
    "The deploy, driven a library at a time."
    FILE = DRIVE_FILE
    KIND = 'drive'
    def __init__(self, root, python=None, env=None, dir=DIR):
        super().__init__(root, python=python, env=env, dir=dir)
        self._rendered = ''
    @staticmethod
    def defaults(root): return parts(root)
    def refresh(self):
        "Re-render the snippets against the current values."
        if self.path.exists(): return self.steps
        key = json.dumps(context(self.root, self.env), sort_keys=True)
        if key != self._rendered:
            self.steps = parts(self.root, self.env, self.python or '')
            self._rendered = key
        return self.steps
    def state(self):
        self.refresh()
        ctx = context(self.root, self.env)
        return super().state() | {'context': ctx, 'packages': sorted({p.package for p in PARTS})}

`Drive` uses rendered parts as its default plan and stores it separately from `Deploy`.

In [ ]:
d = Drive(root, env=env)
st = d.state()
[r['id'] for r in st['steps']], st['packages'], st['needs'], st['configured']

(['image', 'stack', 'tunnel', 'dns', 'server', 'secrets'],
 ['cfeasy', 'dockeasy', 'gheasy', 'vpseasy'],
 ['CLOUDFLARE_API_TOKEN', 'HCLOUD_TOKEN', 'GITHUB_TOKEN'],
 False)

In [ ]:
#| hide
test_eq(d.path, root/'.pullup'/DRIVE_FILE)
test_eq(st['context']['host'], 'app.example.com')
assert 'app.example.com' in d.step('tunnel').argv[2]
env.d['DOMAIN'] = 'other.example.com'
d.refresh()
assert 'other.example.com' in d.step('tunnel').argv[2], 'the plan follows the store until it is saved'
d.save(st['steps'])
frozen = d.step('stack').meta['code']
env.d['DOMAIN'] = 'third.example.com'
d.refresh()
test_eq(d.step('stack').meta['code'], frozen)
test_fail(lambda: d.step('nope'), contains='unknown drive step')

In [ ]:
#| hide
tmp.cleanup()